In [1]:
import numpy as np
from scipy import spatial as sp
from itertools import product, permutations
from matplotlib import pyplot as plt

from transformation import Transformation
from point_cloud import PointCloud


Let $A, B \in \mathbb{R}^k$ be the two point clouds for $k \in \{2, 3\}$ centered at the origin, and let $d_\max = \max\{\text{diam} A, \text{diam} B\}$. Because the distance from the origin to the most extreme point of $A\cup B$ is $< d_\max$, which is invariant to rotation or reflection, we never need to translate $A$ by more than $d_\max$. This implies that the search space $\Omega$ for $k=2$ is (a subset of) $[-d_\max, d_\max]^2 \times [0, 2\pi) \times \{0, 1\}$.

We speed up computing $d_\text{H}\big(T(A), B\big)$ for each $T \in \Omega$ to be $O(n \log n)$ on average by noting that $$d_\text{H}\big(T(A), B\big) = \max\Big\{\overrightarrow{d_\text{H}}\big(T(A), B\big), \overrightarrow{d_\text{H}}\big(B, T(A)\big)\Big\} = \max\Big\{\overrightarrow{d_\text{H}}\big(T(A), B\big), \overrightarrow{d_\text{H}}\big(T^{-1}(B), A\big)\Big\}$$ and setting up a k-d tree once for each space.

In [2]:
def diam(coords):
    hull = sp.ConvexHull(coords)
    hull_coords = coords[hull.vertices]
    candidate_distances = sp.distance.cdist(hull_coords, hull_coords)
    
    return candidate_distances.max()


def dH(A, B, T): # dH(T(A), B)
    return max(A.transform(T).asymm_dH(B),
               B.transform(T.invert()).asymm_dH(A))


def hausDist(A,B):
    m = sp.distance.directed_hausdorff(A, B)[0]
    n = sp.distance.directed_hausdorff(B, A)[0]
    return max(m,n)

In [3]:
%%time
n_dim = 2
n = 100000
n_A, n_B = n, n

# Generate points.
np.random.seed(10)
coords_A, coords_B = [np.random.rand(n, n_dim) for n in [n_A, n_B]]

# Construct the two point clouds.
A, B = (PointCloud(coords) for coords in [coords_A, coords_B])

CPU times: user 140 ms, sys: 26.6 ms, total: 167 ms
Wall time: 168 ms


#### Testing Hausdorff distance between A and B:


In [5]:
%%time
T = Transformation([0], [0, 0], False) # identity
dH(A, B, T)

CPU times: user 532 ms, sys: 18.4 ms, total: 550 ms
Wall time: 242 ms


0.007104935196030845

In [6]:
%%time
hausDist(A.points.coords, B.points.coords)

CPU times: user 4.96 s, sys: 8.54 ms, total: 4.97 s
Wall time: 4.97 s


0.007104935196016974

#### Approximating Euclidean–Hausdorff distance between A and B:


In [10]:
# Compute max diameter.
max_diam = max(map(lambda X: diam(X.points.coords), [A, B]))

# Set up search space.
n_xshifts = n_yshifts = n_thetas = 10
xshifts = np.linspace(-max_diam, max_diam, n_xshifts)
yshifts = np.linspace(-max_diam, max_diam, n_yshifts)
thetas = np.linspace(-max_diam, max_diam, n_thetas, endpoint=False)


In [ ]:
%%time

dEH = np.inf
for xshift, yshift, theta, do_reflect in product(xshifts, yshifts, thetas, range(2)):
    T = Transformation([xshift, yshift], [theta], do_reflect)    
    dEH = min(dH(A, B, T), dEH)
    
print(f'{dEH=}')